# CINEOS Mission One
Package-driven CogVideoX-2B rendering. Reference images are preserved but not consumed by this text-to-video backend; lip-sync is approximate.

In [ ]:
import subprocess

import torch

assert torch.cuda.is_available(), 'CUDA GPU required'
name=torch.cuda.get_device_name(0)
print(name)
assert any(x in name.lower() for x in ('t4','a10','a100','l4','v100','h100')), 'T4 or compatible GPU required'
subprocess.run(['apt-get','update'],check=True); subprocess.run(['apt-get','install','-y','ffmpeg'],check=True)
%pip install -q diffusers transformers accelerate sentencepiece imageio-ffmpeg

In [ ]:
import io
import json
import pathlib
import zipfile

from google.colab import files

uploaded=files.upload(); archive=next(iter(uploaded))
root=pathlib.Path('/content/mission-one'); root.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(uploaded[archive])) as z: z.extractall(root)
package=json.loads((root/'package.json').read_text())
assert len(package['ordered_shot_packages'])==3
print(package['project_id'], package['model_id'])

In [ ]:
import gc
import time

import torch
from diffusers import CogVideoXPipeline
from diffusers.utils import export_to_video

pipe=CogVideoXPipeline.from_pretrained(package['model_id'],torch_dtype=torch.float16).to('cuda')
results=[]; started=time.time()
for shot in package['ordered_shot_packages']:
    generator=torch.Generator('cuda').manual_seed(shot['seed'])
    frames=pipe(prompt=shot['prompt'],negative_prompt=shot['negative_prompt'],num_frames=shot['frame_count'],num_inference_steps=package['inference_steps'],guidance_scale=package['guidance_scale'],generator=generator).frames[0]
    path=root/shot['expected_output']; export_to_video(frames,str(path),fps=package['fps'])
    results.append({'shot_id':shot['shot_id'],'file':path.name,'success':True,'seed':shot['seed']})
    del frames; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# Normalize/concatenate in package order; mix supplied dialogue audio when present.
import subprocess

normalized=[]
for i,shot in enumerate(package['ordered_shot_packages']):
 p=root/shot['expected_output']; n=root/f'normalized-{i}.mp4'; subprocess.run(['ffmpeg','-y','-i',str(p),'-c:v','libx264','-pix_fmt','yuv420p','-r',str(package['fps']),str(n)],check=True); normalized.append(n)
concat=root/'concat.txt'; concat.write_text(''.join(f"file '{p}'\n" for p in normalized))
film=root/'final-film.mp4'; subprocess.run(['ffmpeg','-y','-f','concat','-safe','0','-i',str(concat),'-c','copy',str(film)],check=True)
# Audio manifests may provide an already-timed scene mix; attach it if present.
if package.get('dialogue_audio_manifest'):
 a=root/'audio'/pathlib.Path(package['dialogue_audio_manifest'][0]['path']).name; mixed=root/'final-film-audio.mp4'; subprocess.run(['ffmpeg','-y','-i',str(film),'-i',str(a),'-c:v','copy','-c:a','aac','-shortest',str(mixed)],check=True); mixed.replace(film)
render={'project_id':package['project_id'],'expected_shots':[s['shot_id'] for s in package['ordered_shot_packages']],'shots':results,'final_film':film.name,'model_id':package['model_id'],'hardware':name,'render_time_seconds':time.time()-started}
(root/'render-results.json').write_text(json.dumps(render,indent=2))
verification={'valid':len(results)==3 and film.stat().st_size>0,'missing_shots':[],'reference_conditioning':'unsupported','lip_sync':'approximate_unless_measured','warnings':['Packaged references were not consumed by CogVideoX-2B text-to-video.']}
(root/'verification-report.json').write_text(json.dumps(verification,indent=2))

In [ ]:
from google.colab import files

for name in ('final-film.mp4','render-results.json','verification-report.json'): files.download(str(root/name))